# Lesson 10: The full GPT

**Why:** one round of "talk, then think" took us from 2.31 to 1.90. Real GPTs **repeat** that round many times. Characters gather information, think, gather again with what they now know, think again... Early rounds handle letters, later rounds handle word pieces and words.

A **block** is one round. We stack several. To make stacking work, we need three supporting parts:

| Part | Problem it solves | Plain-English version |
|---|---|---|
| **Residual connection** `x = x + layer(x)` | Deep stacks are hard to train: the learning signal fades as it passes back through many layers | Each layer only **adds a correction** to the information, and the original always passes straight through (a "gradient highway") |
| **LayerNorm** | Numbers drift too big or too small as they pass through many layers | Re-centers each character's numbers to average 0 and spread 1 before each step |
| **Dropout** | The model starts memorizing (Lesson 9's growing train/val gap) | During training, randomly switches off some numbers, so the model can't rely on any single one |

```
idx ─► token emb + position emb
    ─► Block × n_layer:
           x = x + MultiHeadAttention(LayerNorm(x))     talk
           x = x + FeedForward(LayerNorm(x))            think
    ─► LayerNorm ─► Linear ─► logits (B, T, 85)
```

**This is the GPT architecture** (the same design as GPT-2, just much smaller).

## 1. Setup

In [ ]:
from pathlib import Path
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)

text = Path("../data/input.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

batch_size = 32
block_size = 64    # T: the model can now look back 64 characters
n_embd = 64        # C: numbers per character
n_head = 4         # heads per block (each of size 64 / 4 = 16)
n_layer = 4        # how many blocks are stacked
dropout = 0.1      # switch off 10% of numbers during training

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x, y

@torch.no_grad()
def estimate_loss(model, eval_batches=50):
    model.eval()
    result = {}
    for split in ["train", "val"]:
        losses = [model(*get_batch(split))[1].item() for _ in range(eval_batches)]
        result[split] = sum(losses) / len(losses)
    model.train()
    return result

print("ready")

## 2. LayerNorm and dropout up close

**LayerNorm** works on each character's numbers separately: subtract their average, divide by their spread. (It also has two learnable knobs to rescale afterwards, if the model wants.)

$$\text{LayerNorm}(x) = \frac{x - \text{mean}(x)}{\text{spread}(x)}$$

**Dropout** zeroes random numbers during training, and scales the survivors up so the total stays about the same. When we're just measuring or writing text (`eval` mode), it switches itself off.

In [ ]:
x = torch.randn(2, 6) * 5 + 3            # 2 characters, numbers far from 0 and widely spread
ln = nn.LayerNorm(6)
y = ln(x)
print("before: mean", x.mean(dim=1).detach().numpy().round(2), " spread", x.std(dim=1).detach().numpy().round(2))
print("after:  mean", y.mean(dim=1).detach().numpy().round(2), " spread", y.std(dim=1).detach().numpy().round(2))

drop = nn.Dropout(0.5)
ones = torch.ones(10)
drop.train(); print("\ndropout while training:", drop(ones).tolist())
drop.eval();  print("dropout while measuring:", drop(ones).tolist())

## 3. The building blocks

`Head`, `MultiHeadAttention` and `FeedForward` are from Lesson 9, with two small additions: **dropout**, and a **projection** (`proj`) layer that mixes the heads' joined outputs before they're added back onto the residual path.

`Block` is the important new piece. Look at how short it is: the two `x = x + ...` lines **are** the GPT.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        self.last_wei = wei.detach()
        wei = self.dropout(wei)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    # One round of "talk, then think"
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_head, n_embd // n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # talk  (+ residual)
        x = x + self.ffwd(self.ln2(x))    # think (+ residual)
        return x


b = Block(n_embd, n_head)
print("a block keeps the shape:", b(torch.randn(2, 10, n_embd)).shape)

## 4. The GPT

In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(V, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)          # final LayerNorm
        self.lm_head = nn.Linear(n_embd, V)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T))   # (B, T, C)
        x = self.blocks(x)                                      # (B, T, C)
        logits = self.lm_head(self.ln_f(x))                     # (B, T, V)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = GPT()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Train it (about 10 minutes on your laptop)

This is the longest run so far: 2,500 steps, with a progress line every 500 steps. Start it, then take a break.

**Predict:** Lesson 9 reached **1.90**. This model has 4 blocks, a context of 64 characters, and about 10× the parameters. Where will it land: about **1.8**, **1.6** or **1.4**?

Watch the time per 500 steps, and the train/val gap.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = {"step": [], "train": [], "val": []}
start = time.time()

for step in range(2500 + 1):
    if step % 500 == 0:
        l = estimate_loss(model)
        for k, v in [("step", step), ("train", l["train"]), ("val", l["val"])]:
            history[k].append(v)
        print(f"step {step:5d}:  train {l['train']:.3f}   val {l['val']:.3f}   ({time.time() - start:.0f}s)")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

## 6. See it, and read it

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(history["step"], history["train"], "o-", label="train")
plt.plot(history["step"], history["val"], "o-", label="validation")
for y, name in [(2.41, "bigram"), (2.31, "1 head"), (1.90, "4 heads + ffwd")]:
    plt.axhline(y, ls="--", lw=0.8, color="gray"); plt.text(0, y + 0.02, name, fontsize=8, color="gray")
plt.xlabel("step"); plt.ylabel("loss"); plt.ylim(1.2, 3.0); plt.legend(); plt.title("The GPT learning Tesla")
plt.show()

model.eval()
print("--- the GPT writes ---")
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long), 500)[0].tolist()))

## 7. Final check

In [ ]:
final = estimate_loss(model, eval_batches=200)
print(f"GPT val loss: {final['val']:.3f}   (Lesson 9: 1.90)")
assert final["val"] < 1.90, "the stacked GPT should beat Lesson 9"
print("Lesson 10 complete. You've built a GPT.")

## Wrap-up question

In `x = x + self.attn(self.ln1(x))`, what would go wrong if we wrote `x = self.attn(self.ln1(x))` instead (no `x +`), especially with many blocks stacked?